<a href="https://colab.research.google.com/github/hemaswi2008/2026-GenAi-2520030069/blob/Skill/GenAI_exp08.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Experiment 8: Scaled dot-product and multi-head attention from scratch in PyTorch (+ tests against PyTorch)
import math, torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)        # dot-product scores, scaled
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))  # causal / padding mask
    weights = torch.softmax(scores, dim=-1)                    # each row sums to 1
    return weights @ V, weights

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.h, self.d_k = n_heads, d_model // n_heads
        self.W_q, self.W_k, self.W_v, self.W_o = (nn.Linear(d_model, d_model) for _ in range(4))
    def split(self, x):                                        # (B, T, D) -> (B, heads, T, d_k)
        B, T, _ = x.shape
        return x.view(B, T, self.h, self.d_k).transpose(1, 2)
    def forward(self, x, mask=None):
        Q, K, V = self.split(self.W_q(x)), self.split(self.W_k(x)), self.split(self.W_v(x))
        out, w = scaled_dot_product_attention(Q, K, V, mask)
        B, _, T, _ = out.shape
        return self.W_o(out.transpose(1, 2).reshape(B, T, self.h * self.d_k)), w

# ---- Test 1: single-head attention matches PyTorch's built-in ----
Q, K, V = torch.randn(2, 5, 8), torch.randn(2, 5, 8), torch.randn(2, 5, 8)
mine, w = scaled_dot_product_attention(Q, K, V)
assert torch.allclose(mine, F.scaled_dot_product_attention(Q, K, V), atol=1e-5)
assert torch.allclose(w.sum(-1), torch.ones(2, 5))
print("Test 1 passed: scaled dot-product attention == torch, rows of weights sum to 1")

# ---- Test 2: causal mask (decoder / GPT style) ----
causal = torch.tril(torch.ones(5, 5))
mine_c, w_c = scaled_dot_product_attention(Q, K, V, causal)
assert torch.allclose(mine_c, F.scaled_dot_product_attention(Q, K, V, is_causal=True), atol=1e-5)
print("Test 2 passed: causal mask == torch (no token attends to the future)")

# ---- Test 3: multi-head attention matches nn.MultiheadAttention with the same weights ----
x = torch.randn(2, 5, 16)
mha, ref = MultiHeadAttention(16, 4), nn.MultiheadAttention(16, 4, batch_first=True)
with torch.no_grad():
    for lin, W, b in zip([mha.W_q, mha.W_k, mha.W_v], ref.in_proj_weight.chunk(3), ref.in_proj_bias.chunk(3)):
        lin.weight.copy_(W); lin.bias.copy_(b)
    mha.W_o.weight.copy_(ref.out_proj.weight); mha.W_o.bias.copy_(ref.out_proj.bias)
out, weights = mha(x)
assert torch.allclose(out, ref(x, x, x)[0], atol=1e-5)
print("Test 3 passed: multi-head attention == nn.MultiheadAttention | output", tuple(out.shape), "| weights", tuple(weights.shape))

print("\nCausal attention weights (head 0, sequence 0):")
print(torch.round(w_c[0], decimals=2))

Test 1 passed: scaled dot-product attention == torch, rows of weights sum to 1
Test 2 passed: causal mask == torch (no token attends to the future)
Test 3 passed: multi-head attention == nn.MultiheadAttention | output (2, 5, 16) | weights (2, 4, 5, 5)

Causal attention weights (head 0, sequence 0):
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.8400, 0.1600, 0.0000, 0.0000, 0.0000],
        [0.2600, 0.1100, 0.6200, 0.0000, 0.0000],
        [0.4400, 0.0600, 0.0600, 0.4400, 0.0000],
        [0.3200, 0.2500, 0.0800, 0.1800, 0.1800]])
